In [30]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import pickle

In [3]:
fake = pd.read_csv("Fake.csv")
true = pd.read_csv("True.csv")


In [4]:
fake.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [5]:
true.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [6]:
fake.shape


(23481, 4)

In [7]:
true.shape

(21417, 4)

In [8]:
fake['label'] = 0
true['label'] = 1

In [9]:
news = pd.concat([fake,true],axis=0)
news = news.sample(frac=1,random_state=42).reset_index(drop=True)

In [10]:
news.shape

(44898, 5)

In [11]:
news.head()

,title,text,subject,date,label
0,Ben Stein Calls Out 9th Circuit Court: Committ...,"21st Century Wire says Ben Stein, reputable pr...",US_News,"February 13, 2017",0
1,Trump drops Steve Bannon from National Securit...,WASHINGTON (Reuters) - U.S. President Donald T...,politicsNews,"April 5, 2017",1
2,Puerto Rico expects U.S. to lift Jones Act shi...,(Reuters) - Puerto Rico Governor Ricardo Rosse...,politicsNews,"September 27, 2017",1
3,OOPS: Trump Just Accidentally Confirmed He Le...,"On Monday, Donald Trump once again embarrassed...",News,"May 22, 2017",0
4,Donald Trump heads for Scotland to reopen a go...,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",politicsNews,"June 24, 2016",1


In [12]:
news.isnull().sum()

title      0
text       0
subject    0
date       0
label      0
dtype: int64

In [13]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+","",text)
    text = text.translate(str.maketrans("","",string.punctuation))
    text = re.sub(r"\s+"," ",text)
    return text

In [14]:
news['content'] = news['title']+" "+news['text']

In [15]:
news['content'] = news['content'].apply(clean_text)

In [16]:
news['content'].head()

0    ben stein calls out 9th circuit court committe...
1    trump drops steve bannon from national securit...
2    puerto rico expects us to lift jones act shipp...
3     oops trump just accidentally confirmed he lea...
4    donald trump heads for scotland to reopen a go...
Name: content, dtype: object

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(stop_words="english",max_df=0.7)

In [18]:
x = news["content"]
y = news["label"]

In [19]:
x = vectorizer.fit_transform(x)
x.shape

(44898, 223631)

In [20]:
print(vectorizer.get_feature_names_out()[:20])

['00' '000' '0000' '000000017' '000004' '000048' '000063sz' '00007'
 '00009' '0001' '00011' '00018' '000270ks' '0005' '0006' '00075' '00076'
 '0009' '000938sz' '000dillon000']


In [21]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

In [22]:
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [23]:
model = LogisticRegression(max_iter=1000)
model.fit(x_train,y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [24]:
y_pred = model.predict(x_test)
print(y_pred[:20])

[0 1 1 0 1 1 0 0 1 0 0 0 1 1 1 0 1 1 0 1]


In [29]:
accuracy = accuracy_score(y_test,y_pred)
print(f"Model Accuracy:{accuracy*100: .2f}%")

Model Accuracy: 98.41%


In [26]:
cm = confusion_matrix(y_test,y_pred)
print(cm)

[[4633   77]
 [  66 4204]]


In [27]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.99      0.98      0.98      4710
           1       0.98      0.98      0.98      4270

    accuracy                           0.98      8980
   macro avg       0.98      0.98      0.98      8980
weighted avg       0.98      0.98      0.98      8980



In [31]:
with open("model.pkl","wb") as file:
    pickle.dump(model, file)

In [32]:
with open("vectorizer.pkl","wb") as file:
    pickle.dump(vectorizer, file)